# LLaVA-based Concept Verification over CLIP Scores
Modular helpers to verify top-K concepts per image using a yes/no LLaVA query, gate CLIP scores, and compute mismatch stats.

In [11]:
import os
from typing import List, Tuple, Optional, Dict, Any, Iterable
import torch
import pandas as pd
from PIL import Image


In [3]:
# load from numpy
import numpy as np

concept_feat = torch.load('../exp/asso_opt/CIFAR10/CIFAR10_2shot_fac/concepts_feat_ViT-L-14.pth')
img_feat = torch.load('../datasets/CIFAR10/splits/img_feat_train_2_00_ViT-L-14.pth')

concept_feat.shape, img_feat.shape

(torch.Size([7099, 768]), torch.Size([20, 768]))

In [12]:
test_img_feat = torch.load('../datasets/CIFAR10/splits/img_feat_test_00_ViT-L-14.pth')

test_img_feat.shape

torch.Size([10000, 768])

In [7]:
torch.load('../exp/asso_opt/CIFAR10/CIFAR10_2shot_fac/select_idx.pth').shape


torch.Size([500])

In [3]:
scores = concept_feat @ img_feat.T
scores.shape

torch.Size([7099, 20])

In [4]:
def get_topk_concepts(scores, concept_texts, k=5):
    # scores: [num_concepts, num_images]
    topk_per_image = []

    num_images = scores.shape[1]

    for i in range(num_images):
        image_scores = scores[:, i]  # all concepts for image i

        topk_vals, topk_idx = torch.topk(image_scores, k)

        concepts = [concept_texts[idx] for idx in topk_idx]

        topk_per_image.append({
            "image_idx": i,
            "concept_indices": topk_idx.tolist(),
            "concept_texts": concepts,
            "scores": topk_vals.tolist()
        })

    return topk_per_image

In [10]:
import numpy as np

concept_texts = np.load("../exp/asso_opt/CIFAR10/CIFAR10_2shot_fac/concepts_raw_selected.npy", allow_pickle=True)
concept_texts[:5]

array(['"points" include the ears, face, paws, and tail',
       '/word> sentence>the overall impression is one of power, reliability, and a readiness for work./sentence>',
       'a behemoth of polished steel',
       'a burst of color against the metallic sheen',
       'a certain elegance and elongation characterize its bodily shape'],
      dtype='<U220')

In [6]:
topk_data = get_topk_concepts(scores, concept_texts, k=5)

In [8]:
for item in topk_data[:3]:
    print("\nImage:", item["image_idx"])
    for c, s in zip(item["concept_texts"], item["scores"]):
        print(f"{c} → {s:.3f}")


Image: 0
hint at aircraft's ability to conquer vast distances → 80.830
adheres to the grander pattern of global commerce → 73.960
a sense of precision and modern aviation technology → 72.394
designed for aerodynamic efficiency and giving the aircraft an assertive presence → 71.678
spoke of speed, efficiency, and a touch of aspirational flight → 70.702

Image: 1
flicked with an almost imperceptible motion → 85.358
hint at aircraft's ability to conquer vast distances → 83.663
hinting at strong, soaring flight → 83.309
indicative of its engineered purpose → 83.030
visually dynamic effect when in flight → 79.703

Image: 2
hinting at its forward momentum → 85.270
starts subtly near the front wheel well and widens as it moves towards the rear → 84.697
hinting at underlying speed and sportiness → 84.273
hinting at high speeds and maneuverability → 83.441
possesses a compact, somewhat rounded body → 83.389


In [9]:
import pickle

with open("../datasets/CIFAR10/splits/class2images_train.p", "rb") as f:
    class2images = pickle.load(f)

In [10]:
print(type(class2images))
print(list(class2images.keys())[:5])
print(class2images['airplane'][:5])

<class 'dict'>
['airplane', 'automobile', 'bird', 'cat', 'deer']
['stealth_bomber_s_000554.png', 'twinjet_s_000663.png', 'monoplane_s_000895.png', 'twinjet_s_000591.png', 'fighter_aircraft_s_000876.png']


In [13]:
print(len(class2images['airplane']))

4500
